Make a heatmap of activity in the few second preceding the homing, sorted by pref tuning in the homing

In [1]:
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

experiments_objects = {"JAL4_3rdSept": JAL4_3rdSept,
        "JAL4_19thSept": JAL4_19thSept,
        "JAL4_28aug": JAL4_28aug,
        "JAL4_11thSept": JAL4_11thSept,
        "JAL5_8thSept": JAL005_8thSept,
        "JAL5_21stSept": JAL005_21stSept,
        "JAL6_28mar": JAL6_28mar, 
        "JAL6_flip4_21mar": JAL6_flip4_21mar, 
        "JAL6_flip3_18mar": JAL6_flip3_18mar, 
        "JAL6_flip5_25mar": JAL6_flip5_25mar, 
        "JAL7_sesh8_9apr": JAL7_sesh8_9apr, 
        "JAL7_flip5_22mar": JAL7_flip5_22mar, 
        "JAL7_flip2_12mar": JAL7_flip2_12mar, 
        "JAL7_sesh9_16apr": JAL7_sesh9_16apr, 
        "JAL7_23apr": JAL7_23apr,
        "JAL8_flip1_25apr": JAL8_flip1_25apr, 
        "JAL8_flip2_29apr": JAL8_flip2_29apr, 
        "JAL8_flip3_7may": JAL8_flip3_7may, 
        "JAL8_14may": JAL8_14may, 
        "JAL8_flip4_10may": JAL8_flip4_10may}

In [2]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.process.process import Process
from JR_test_scripts.escape.escape_utils import load_homing, load_hdir_cells, load_significant_cells
from JR_test_scripts.replay.BayesianReplay_funcs import bayesian_decoder, calculate_custom_replay_score, calculate_radon_score, calculate_linear_weighted_correlation
import matplotlib.gridspec as gridspec

import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
from scipy.ndimage import gaussian_filter1d
from scipy.stats import zscore
import matplotlib
from syd import Viewer

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 30
matplotlib.rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded in PDF

%matplotlib inline

In [3]:
"""Load the data"""
c_names = ['shelter_only', 'barrier', 'flipped_barrier']
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
explore_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
homie_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
save_path = make_directory("Z:/Jasmine_Laurence/summary_plots/Replay/PreHomingSorted/DecodeHomings/")
tuning_data = '_25bins' # '' or '_50bins' or '_25bins
cond_colors = ['#228B22','#FF8C00','#008B8B']
homing_color = '#6A0DAD'
escape_color = '#E63946'

In [4]:
"""Load data and Extract the homing and escape x and y positions"""

# hard coded vars for computing replay score
max_fract = 100 # max %escape
# Define parameter search ranges (adjust based on expected replay speeds/positions)
# Velocity range (e.g., -10 m/s to +10 m/s, 101 steps) -> -1000 cm/s to 1000 cm/s
V_range = [-50, 50, 51] # in %escape per second
# Starting position range (e.g., full track length, 100 steps)
rho_range = [0, max_fract, 100] # in %escape
# Define parameters
fract_thresh = 10.0 # Threshold of fraction of route in percent
time_bin_width = 0.025  # 25 ms for a 40Hz frame rate

class PreHoming(Viewer):
    def __init__(self):

        self.nickname = ''
        self.homing = []
        self.condition = ''
        self.add_selection('Session', value = list(experiments_objects.keys())[7], options = list(experiments_objects.keys()))
        self.add_selection('Condition', value = 'shelter_only', options = conditions)
        self.add_integer('Time_s', value = 3, min = 1, max = 5)
        self.add_integer('Homing idx', value = 0, min = 0, max = 100)
        self.add_button('savefig', label='Save Figure', callback=self.save_fig, replot = False)

    def load_data(self, state):
        
        session = Process(experiments_objects[state['Session']]).load_session()
        base_path = os.path.join(session.base_path, session.processed_path)

        # matrix
        frame_by_cluster_matrix = np.load(base_path + "\\" + "frame_by_good_cluster_matrix.npy")

        # full x and y pos
        video_df = pl.read_csv(os.path.join(base_path, "full_video_dataframe.csv"))
        y_pos = video_df["mouse_y_position"].to_numpy()
        x_pos = video_df["mouse_x_position"].to_numpy()
        bar = video_df["barrier_present"].to_numpy()
        barflip = video_df["barrier_flipped"].to_numpy()

        # load homings
        h_on, h_off, homing_bool = load_homing(session, int(np.amax(np.unique(video_df['frames'].to_numpy()))))

        # booleans for homing+escape and explore
        # TODO this is a hack into the escapes to remove the 1s of pause before they start running
        escape = video_df['EscapePeriod'].to_numpy()
        escape_bool = np.full_like(escape, False)
        start = np.where(np.diff(escape.astype(int)) == 1)[0]
        end = np.where(np.diff(escape.astype(int)) == -1)[0]
        for s, e in zip(start, end):
            escape_bool[s+40:e] = True

        h_e_bool = (homing_bool | escape_bool) & (video_df['OutofshelterIdx'].to_numpy())
        full_e_bool = (escape_bool) & (video_df['OutofshelterIdx'].to_numpy())
        self.cond = np.zeros(len(bar))
        self.cond[bar] += 1
        self.cond[barflip] += 1

        X = x_pos[h_e_bool]
        Y = y_pos[h_e_bool]
        h_cond = self.cond[h_e_bool]

        # cell tuning pref
        self.nickname = experiments_objects[state['Session']].nick_name + '_' + experiments_objects[state['Session']].experiment_date + '_' + 'escape'
        data = np.load(homie_path + self.nickname + '_ProperTuning' + tuning_data + '.npz')
        self.preferred_tuning = data['params_full'][:,:,1]
        self.tuning_curve = data['fr_full']

        # starts of homings & escapes
        starts = np.where(np.diff(h_e_bool.astype(int)) > 0)[0] + 1
        ends = np.where(np.diff(h_e_bool.astype(int)) < 0)[0] + 1
        homie_lengths = ends - starts
        counter, h_start, e_start = 0, np.full(len(homie_lengths), 0), np.full(len(homie_lengths)+1, False)
        for i, h in enumerate(homie_lengths):
            if full_e_bool[starts[i]]:
                e_start[i] = True # a bool that tells us which one of the h_starts are escapes
            h_start[i] = counter 
            counter += h
        h_start = np.append(h_start, len(h_cond)) # a list of the start indices of homings + escapes in the homing/escape time period
        e_bool = full_e_bool[h_e_bool] # a boolean that tells us which periods of the homing/escape are escapes

        # where did the mouse start and end
        y_start = Y[h_start[:-1]]
        y_end = Y[h_start[1:]-1]
        long_homie_bool = np.full(len(h_start[:-1]), False)
        homie_id = np.full(len(X), 0) # a vector that increases with each homing
        for h, (s,e) in enumerate(zip(y_start, y_end)):
            homie_id[h_start[h]:h_start[h+1]] = h
            if (s < 512) & (e > 700):
                long_homie_bool[h] = True

        # find the times before the homie starts
        homie = np.where(np.diff(h_e_bool.astype(int)) > 0)[0] + 1
        self.prebool = np.full(len(h_e_bool), False)
        for idx, i in enumerate(homie):
            if long_homie_bool[idx]:
                # self.prebool[i-(state['Time_s']*40):i] = True
                # run this one if you want a sanity check of the homings
                self.prebool[i:i+(state['Time_s']*40)] = True
        
        self.sig_cells = load_significant_cells(experiments_objects[state['Session']], case = "either_tuned", tuning_data = tuning_data)
        hdir = load_hdir_cells([experiments_objects[state['Session']]], [state['Session']])
        self.sig_cells[hdir,:] = np.full(3, False)

        # filter only the cells that are not hdir and are sig in at least one condition to either dist or escape
        self.fcm = frame_by_cluster_matrix
        # self.fcm = gaussian_filter1d(frame_by_cluster_matrix, 2, axis = 0)

    def condition_data(self, state):
        self.this_cond = [x for x in range(len(conditions)) if conditions[x] == state['Condition']][0]
        # pull out neural activity for this condition
        neural_matrix = self.fcm[self.prebool & (self.cond == self.this_cond),:] # select the homing and escape periods for the condition
        self.neural_matrix = neural_matrix[:,self.sig_cells[:,self.this_cond]] # select the cells that are sig in the condition

        # sort the cells by preferred tuning
        isort = np.argsort(self.preferred_tuning[self.sig_cells[:,self.this_cond],self.this_cond])
        self.neural_matrix = self.neural_matrix[:,isort]
        self.tuning_curve_sort = self.tuning_curve[self.this_cond,self.sig_cells[:,self.this_cond],:]
        self.tuning_curve_sort = self.tuning_curve_sort[isort,:]

        self.neural_matrix_z = zscore(self.neural_matrix, axis = 0)
        self.condition = state['Condition']
        self.homing = []

    def decode(self, state):
        # firing_rate_maps: A 2D array or list of arrays. Shape: (n_neurons, n_position_bins). Contains the average firing rate of each neuron i for each position bin x. These are the "templates".
        # spike_counts: A 2D array. Shape: (n_time_bins, n_neurons). Contains the number of spikes n_i detected for each neuron i within each time bin t of the candidate event.
        # occupancy_map: A 1D array. Shape: (n_position_bins,). Contains the normalized probability P(x) of the animal being in each position bin x, derived from overall session behavior.
        # time_bin_width (τ): A float. The duration of each time bin in seconds (e.g., 0.02 for 20ms).
        # n_neurons (N): An integer. The total number of neurons used for decoding.
        # n_position_bins: An integer. The number of spatial bins used to discretize the environment.
        # n_time_bins: An integer. The number of time bins in the candidate event window.

        firing_rate_map = self.tuning_curve_sort
        self.spike_counts = self.neural_matrix[state['Homing idx']*40*state['Time_s']:(state['Homing idx']+1)*40*state['Time_s'],:]
        # set up some variables
        n_position_bins = firing_rate_map.shape[1]
        n_time_bins = self.spike_counts.shape[0]
        
        # Define position bins (replace with your actual bins)
        position_bin_edges = np.linspace(0, max_fract, n_position_bins + 1)

        # create the prior
        # occupancy_counts = np.bincount(discretized_escape.astype(int), minlength=n_position_bins)
        # occupancy_map = occupancy_counts / np.sum(occupancy_counts)
        occupancy_map = np.ones(n_position_bins) / n_position_bins # Uniform prior

        self.posterior = bayesian_decoder(firing_rate_map, self.spike_counts, occupancy_map, time_bin_width, n_time_bins, n_position_bins)
        # radon_score, radon_angle = calculate_radon_score(self.posterior)
        self.linear_corr = calculate_linear_weighted_correlation(self.posterior)
        self.R_max, self.V_max, self.rho_max, self.R_map = calculate_custom_replay_score(self.posterior, time_bin_width, position_bin_edges, fract_thresh, V_range, rho_range)
        self.homing = state['Homing idx']

    def plot(self, state):

        if self.nickname != experiments_objects[state['Session']].nick_name + '_' + experiments_objects[state['Session']].experiment_date + '_' + 'escape':
            self.load_data(state)
        if self.condition != state['Condition']:
            self.condition_data(state)
        if self.homing != state['Homing idx']:
            self.decode(state)

        # Set up figure with gridspec 
        fig = plt.figure(figsize=(30, 30))
        gs = gridspec.GridSpec(nrows=2, ncols=3)

        # Create the heatmap
        ax_heatmap = fig.add_subplot(gs[0,:])
        ax_heatmap.imshow(self.neural_matrix_z.T, vmin=-.5, vmax=1.5, cmap="gray_r", aspect="auto", interpolation="none",
                          extent=(0, self.neural_matrix_z.shape[0]/40, 0, self.neural_matrix_z.shape[1]))
        ax_heatmap.set_title(f"Neural activty {state['Time_s']} s before homings")
        ax_heatmap.set_xlabel('Time (s)')

        h_start = np.arange(0, self.neural_matrix_z.shape[0]/40, (state['Time_s']))
        for start, i in enumerate(h_start):
            ax_heatmap.axvline(x=i, color=homing_color, linestyle='--')

        ax_heatmap.set_ylabel("Neurons")

        # add plots to look at individual pre-homing periods and if we can decode the trajectory
        ax = fig.add_subplot(gs[1,0])
        im = ax.imshow(zscore(self.spike_counts, axis = 0).T, 
                       vmin=-.5, vmax=1.5, aspect = 'auto', cmap = 'Greys', interpolation = 'none',
                       extent=(state['Homing idx']*state['Time_s'], (state['Homing idx']+1)*state['Time_s'], 0, self.neural_matrix_z.shape[1]))
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Neurons')
        ax.set_title('Neural activity \n in homing&escapes')
        fig.colorbar(im, ax=ax, label='z-score')

        ax = fig.add_subplot(gs[1,1])
        im = ax.imshow(self.posterior.T, aspect='auto', origin='lower', interpolation='nearest',
                       extent=(0, self.posterior.shape[0]*time_bin_width, 0, max_fract))
        ax.set_xlabel('Time Bin (s)')
        ax.set_ylabel(f'Position Bin % escape')
        ax.set_title('Posterior Probability' + '\n' + f"Linear Correlation: {self.linear_corr:.2f}")
        fig.colorbar(im, ax=ax, label='Posterior Probability')
        plt.tight_layout()

        ax = fig.add_subplot(gs[1,2])
        im = ax.imshow(self.R_map.T, aspect='auto', origin='lower',
                    extent=[V_range[0], V_range[1], rho_range[0], rho_range[1]], cmap='viridis')
        fig.colorbar(im, ax=ax, label = 'Replay Score R(V, ρ)')
        ax.scatter([self.V_max], [self.rho_max], color='red', marker='x', s=100, label=f'Max R ({self.R_max:.3f})')
        ax.set_xlabel('Velocity V (%/s)')
        ax.set_ylabel(r'Starting Position ρ %')
        ax.set_title(f"Replay Score: {self.R_max:.4f}" + '\n' + f"Starting Position: {self.rho_max:.2f} %" + '\n' + f"Velocity: {self.V_max:.2f} %/s")
        ax.legend()

        self.fig = fig

        return fig
    
    """Callback functions"""
    def save_fig(self, state):
        # Save the figure
        self.fig.savefig(os.path.join(save_path, f"{state['Session']}_pattern_heatmap_{state['Condition']}_Homing{state['Homing idx']}.png"), dpi=300)


In [5]:
%matplotlib inline
env = "notebook"
viewer = PreHoming().deploy(env=env)

2025-05-20 20:47:26.307 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2025-05-20 20:48:07.722 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...
2025-05-20 20:48:12.597 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
